# An AI Agent that plans out your weekend.

Agent can plan out your weekend following weather you want, how you want to commute, or which events you want to go to!

In [ ]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip install geopy

Download the required libraries for our AI Agent

In [ ]:
import requests
import json
import re
import os
from datetime import date
from geopy.geocoders import Nominatim
from groq import Groq
from google.colab import userdata

Include the environment variables.

You may create your own on TicketMaster, Google Cloud, and Groq.

In [ ]:
os.environ["TICKET_MASTER_API_KEY"] = userdata.get("TICKET_MASTER_API_KEY")
os.environ["GOOGLE_ROUTES_API_KEY"] = userdata.get("GOOGLE_ROUTES_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

Create a function to lookup weather using Open Meteo.

Format the output from Open Meteo to only access what we need.

In [ ]:
WEATHER_LOOKUP_OUTLOOK = {1,3,7,14,16}
WMO_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Icy fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Heavy drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    77: "Snow grains",
    80: "Slight showers",
    81: "Moderate showers",
    82: "Violent showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}
def weather_lookup(city: str, forecast_days: str) -> str:
  forecast_days = int(forecast_days)
  if forecast_days not in WEATHER_LOOKUP_OUTLOOK:
    return {"Error": "Forcast days not in allowed range."}
  geolocator = Nominatim(user_agent="my_app")
  location = geolocator.geocode(city)
  lat, lon = location.latitude, location.longitude
  params = {
    "latitude": lat,
    "longitude": lon,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode",
    "forecast_days": forecast_days,
  }
  req = requests.get(f"https://api.open-meteo.com/v1/forecast", params=params)
  parsedReq = req.json()["daily"]
  allForecasts = []
  for i in range(forecast_days):
    forecast = {
        "date": parsedReq["time"][i],
        "temperature_max": parsedReq["temperature_2m_max"][i],
        "temperature_min": parsedReq["temperature_2m_min"][i],
        "precipitation_sum": parsedReq["precipitation_sum"][i],
        "weatherDescription": WMO_CODES[parsedReq["weathercode"][i]]
    }
    allForecasts.append(forecast)
  return str(allForecasts)

Create a function to search for events using the TicketMaster API

Format the output from TicketMaster API to only access what we need for our agent.

In [ ]:
def events_search(city: str) -> str:
  req = requests.get(f"https://app.ticketmaster.com/discovery/v2/events.json?city={city}&apikey={os.environ["TICKET_MASTER_API_KEY"]}")
  parsedReq = req.json()
  maskedEventsList = []
  for event in parsedReq["_embedded"]["events"]:
    maskedEvent = {"name": event["name"], "type": event["type"], "dates": event["dates"], "venues": []}
    for venue in event["_embedded"]["venues"]:
      maskedEvent["venues"].append({"venue_name": venue["name"], "venue_city": venue["city"]["name"]})
    maskedEventsList.append(maskedEvent)
  return str(maskedEventsList)

Create a function to search for routes between our location and the destination.

We may choose the 5 specified travel modes based on AI Agent's input.

In [ ]:
METERS_PER_MILE = 1609
def routes_search(origin: str, dest: str, travel_mode: str, departure_time_iso: str = None) -> str:
  if travel_mode not in {"DRIVE", "WALK", "BICYCLE", "TWO_WHEELER", "TRANSIT"}:
    return {"Error": "Travel mode not allowed"}
  if travel_mode == "TRANSIT" and not departure_time_iso:
    return {"Error": "Transit mode requires an arrival or departure time."}
  url = "https://routes.googleapis.com/directions/v2:computeRoutes"
  headers = {
      "Content-Type": "application/json",
      "X-Goog-Api-Key": os.environ["GOOGLE_ROUTES_API_KEY"],
      "X-Goog-FieldMask": "routes.duration,routes.distanceMeters"
  }
  payload = {
      "origin": {
          "address": f"{origin}"
      },
      "destination": {
          "address": f"{dest}"
      },
      "travelMode": f"{travel_mode}",
      "routingPreference": "TRAFFIC_AWARE"
  }

  if travel_mode in {"TRANSIT", "WALK", "BICYCLE"}:
    del payload["routingPreference"]
  if departure_time_iso:
    payload["departureTime"] = departure_time_iso
  if travel_mode == "TRANSIT":
    payload["transitPreferences"] = {
        "allowedTravelModes": ["BUS", "SUBWAY", "TRAIN"],
        "routingPreference": "LESS_WALKING"
    }
  else:
    payload["computeAlternativeRoutes"] = True
  res = requests.post(url, json=payload, headers=headers)
  routesRes = []
  for route in res.json()["routes"]:
    routesRes.append({
      "miles": route["distanceMeters"] / METERS_PER_MILE,
      "durationHours": str(int(route["duration"][:-1]) / 3600)[:4] + "h"
      })
  return str(routesRes)


We must create concrete descriptions of the tools our agent is allowed to use. It is important to describe to the agent the type of input and the expected output per tool.

We must also specify how the parameters must be formatted for function calls.

In [ ]:
TOOLS_DESCRIPTION = """
weather_lookup(city: str, forecast_days: str) -> str
  Args:
  - city (the city you want the weather in, e.g., 'New York')
  - forecast_days (the number of days you want to forecast weather for, e.g., '7')
    NOTE: forecast_days only allows '1', '3', '7', '14', '16' as inputs.

  Output:
  - An array of weather data objects for each forecast days, e.g. [{'date': '...', 'temperature_max': ..., 'temperature_min': ..., 'precipitation_sum': ..., 'weatherDescription': '...'},...]

events_search(city: str) -> str:
  Args:
  - city (the city you wnat to look up events in, e.g., 'Los Angeles')
  Output:
  - An array of event objects in the city. This object includes numerous data items, e.g.,
  [{'name': '...',
    'type': 'event',
    'dates': {'start': {'localDate': '...',
      'localTime': '...',
      'dateTime': '...',
      'dateTBD': ...,
      'dateTBA': ...,
      'timeTBA': ...,
      'noSpecificTime': ...},
    'timezone': '...',
    'status': {'code': '...'},
    'spanMultipleDays': ...},
    'venues': [{'venue_name': '...',
      'venue_city': '...'}]},...],

routes_search(origin: str, dest: str, travel_mode: str, departure_time_iso: str) -> str:
  Args:
    - origin (the place you want to start your travel, e.g. 'Madison Square Park' or 'Phoenix, AZ' or '87 White St, NY, NY')
    - destination (the place you want to go to, e.g., 'Times Square', 'Salt Lake City', 'Madison Square Garden')
    - travel_mode (the way you want to travel, e.g. "DRIVE", "TRANSIT", "BICYCLE")
        NOTE: You can only choose "DRIVE", "WALK", "BICYCLE", "TWO_WHEELER", "TRANSIT" as transit modes.
    - departure_time_iso (the time you want to depart from your origin location, e.g., '2026-5-20T15:00:00Z')
        NOTE: departure_time_iso is an optional field. You must specify departure_time_iso in ISO format if given.
  Output:
    - An array of routes by which you can arrive at the destination with the given travel mode.
    This includes the amount of miles you must travel and the time it takes in hours.
    e.g.,
    [{'miles': ..., 'durationHours': '...'},
      {'miles': ..., 'durationHours': '..'},
      {'miles': ..., 'durationHours': '...'}]
"""

In [ ]:
TOOLS = {
    "weather_lookup": weather_lookup,
    "events_search": events_search,
    "routes_search": routes_search
}

This is the core ReAct loop, i.e, Reasoning and Action.

The agent continues to reason about the next tool to use until obtains the final answer.

In [ ]:
SYSTEM_PROMPT = f""" You are a ReAct agent that solves problems step by step. Today is {date.today()}

Tools Available:
{TOOLS_DESCRIPTION}

Format (follow exactly):
Thought: <reasoning>
Action: {{"tool": "<tool_name>", "inputs": ["<input1>", "<input2>"]}}
After Action, STOP. The system replies with:

Observation: <result>

Continue with another Thought/Action, or finish:

Thought: I now know the final answer.
Final Answer: <answer>

Rules:
- One Thought + Action per turn, then wait.
- Action must be one of: {list(TOOLS.keys())}
- Action Input is a plain string with approrpiate spaces when needed (no quotes).
- Never invent Observations.
- Do not use function calling. Only respond in plain text using the Thought/Action format.
"""

Parsing the Agent's string output into tuple for easier function calls.

In [ ]:
def parse_response(text: str):
    if "Final Answer:" in text:
        return ("final", text.split("Final Answer:")[-1].strip())

    if m := re.search(r"Action:\s*(\{.+\})", text):
        data = json.loads(m.group(1))
        return ("action", data["tool"], data["inputs"])

    return ("error", "Could not parse.")

Function for ReAct loop, using Llama model.

We may configure the maximum step size by changing the default 'steps' value in the function header or inputing our own arguement.

In [ ]:
def ask_agent(query: str, steps: int = 10) -> str:
  client = Groq(api_key=os.environ["GROQ_API_KEY"])
  MODEL = "llama-3.3-70b-versatile"
  messages = [
      {"role": "system",
       "content": f"{SYSTEM_PROMPT}"
      },
      {"role": "user",
       "content": f"{query}"}
  ]
  for i in range(steps):
    res = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0,
        stop=["Observation:"],
    )
    out = res.choices[0].message.content.strip()
    print(out)
    messages.append({"role": "assistant", "content": out})
    parsedRes = parse_response(out)
    if parsedRes[0] == "final":
      return parsedRes[1]
    if parsedRes[0] == "error":
      return parsedRes[0]
    if parsedRes[1] not in list(TOOLS.keys()):
      return "Error: Unspecified tool called!"
    _, tool, inputData = parsedRes
    toolRes = TOOLS[tool](*inputData)
    messages.append({"role": "user", "content": f"Observation: {toolRes}"})
  return "Step limit exceeded."

Sample use case. It is important to note that we can configure out input in any format, as long as it pertains to events, weather, and/or commute!

Note that shorter and/or simpler prompts are quicker to execute.

In [ ]:
import time
start = time.time()
print(ask_agent("I'm in Edinburgh and want to attend an outdoor event in London this weekend, but only if it won't rain. If it will rain, find me an indoor event instead. How long is the drive and should I leave Friday night or Saturday morning to make it on time?"))
print(time.time() - start)

Thought: First, I need to check the weather forecast for London this weekend to determine if it will rain, which will help me decide whether to look for an outdoor or indoor event.
Action: {"tool": "weather_lookup", "inputs": ["London", "3"]}
Thought: The weather forecast indicates that it will be drizzling in London over the weekend, so I should look for an indoor event instead of an outdoor one. I will search for events in London to find a suitable indoor event.
Action: {"tool": "events_search", "inputs": ["London"]}
Thought: I found an indoor event, "Harry Potter and the Cursed Child - Parts 1 & 2 Wed 14:00 & 19:07", at the Palace Theatre in London on May 20. Now, I need to calculate the driving time from Edinburgh to London to determine if I should leave on Friday night or Saturday morning to make it on time for the event.
Action: {"tool": "routes_search", "inputs": ["Edinburgh", "Palace Theatre, London", "DRIVE", "2026-05-20T06:00:00Z"]}
Thought: The driving time from Edinburgh to